# Python Prescriptive Analytics - 2026-10-28 - YOUR_STUDENT_ID

Replace YOUR_STUDENT_ID here and in the setup cell. Open this notebook in VCASTLE JupyterLab. Confirm the Excel path below, then run all cells. Capture your own screenshots with your student ID, date, and time visible and place them in your lab report template. This notebook adapts the supplied guided practice to the Airbnb assignment. No lab report template was attached.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import minimize, differential_evolution
from pathlib import Path

student_id = "YOUR_STUDENT_ID"
file_path = Path(r"C:\Files\Excel\Prescriptive_PythonR\AirbnbListings_v4.xlsx")
# Alternatively, put the workbook beside this notebook.
if not file_path.exists():
    file_path = Path("AirbnbListings_v4.xlsx")

def header(title):
    display(Markdown(f"### {title} - {student_id} - {datetime.now():%Y-%m-%d %H:%M:%S}"))

def plot_title(title):
    return f"{title}\n{student_id} - {datetime.now():%Y-%m-%d %H:%M:%S}"


## 1–2. Load, describe, and clean the dataset
Missing target prices are removed rather than invented. Availability must be between 0 and 365 days. Invalid availability is treated as missing and replaced with the median valid availability. The original dataframe is preserved. The other columns are not model predictors.

In [ ]:
header("Load and Explore the Dataset")
airbnb_df = pd.read_excel(file_path)
features = ["minimum_nights", "number_of_reviews", "availability_365"]
print("Original shape:", airbnb_df.shape)
display(airbnb_df.head())
display(airbnb_df.describe())
display(airbnb_df.isna().sum().rename("Original null count"))

clean_df = airbnb_df.copy()
for column in features + ["price"]:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")
missing_prices = int(clean_df["price"].isna().sum())
invalid_availability = clean_df["availability_365"].notna() & ~clean_df["availability_365"].between(0, 365)
invalid_availability_count = int(invalid_availability.sum())
clean_df.loc[invalid_availability, "availability_365"] = np.nan
clean_df = clean_df.dropna(subset=["price"]).copy()
feature_medians = clean_df[features].median()
clean_df[features] = clean_df[features].fillna(feature_medians)
print(f"Removed {missing_prices} rows with missing price.")
print(f"Replaced {invalid_availability_count} invalid availability values with the valid median.")
display(feature_medians.rename("Feature median"))
print("Cleaned shape:", clean_df.shape)
display(clean_df[features + ["price"]].isna().sum().rename("Remaining model-column nulls"))


## 3–4. Feature importances
As in the guided practice, fit the model on the cleaned dataset. A fixed random seed makes results reproducible. These are model associations, not evidence that changing a feature causes a price change; no out-of-sample accuracy is claimed.

In [ ]:
header("Feature Importances")
X = clean_df[features]
y = clean_df["price"]
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)
feature_importance_df = pd.DataFrame({"Feature": features, "Importance": model.feature_importances_}).sort_values("Importance", ascending=False)
display(feature_importance_df)
plt.figure(figsize=(9, 5))
plt.barh(feature_importance_df["Feature"], feature_importance_df["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title(plot_title("Feature Importances"))
plt.tight_layout()
plt.show()


## 5. Feature sensitivity analysis
Following the guided practice, vary one predictor across its observed range for all records, preserve the other two predictors, and average the predictions. Produce one graph per feature.

In [ ]:
for feature in features:
    header(f"Sensitivity Analysis: {feature}")
    feature_values = np.unique(np.rint(np.linspace(X[feature].min(), X[feature].max(), 100))).astype(int)
    results = []
    for value in feature_values:
        X_temp = X.copy()
        X_temp[feature] = value
        results.append(model.predict(X_temp).mean())
    plt.figure(figsize=(9, 5))
    plt.plot(feature_values, results)
    plt.xlabel(feature)
    plt.ylabel("Average predicted price ($)")
    plt.title(plot_title(f"Sensitivity Analysis of {feature}"))
    plt.tight_layout()
    plt.show()


## 6. Define and solve the optimization problem

| Component | Definition |
|---|---|
| Objective | Maximize predicted listing price, not revenue or profit. |
| Decision variables | minimum_nights, number_of_reviews, availability_365 |
| Constraints | Integer values within each cleaned feature’s observed minimum and maximum; availability is also limited to 0–365. |
| Guided-practice comparison | L-BFGS-B starts at mean feature values. A random forest is piecewise constant, so this method can stop without finding a useful optimum. |
| Main search | Seeded differential evolution over rounded integer feature values, compared against observed feature combinations. Report the best value found, not a proven global maximum. |

Reviews are a hypothetical input, not something a host can directly set. Combining observed feature ranges can also produce combinations unlike actual listings.

In [ ]:
header("Solve the Optimization Problem")
bounds = [(int(X[col].min()), int(X[col].max())) for col in features]
baseline = X.mean()
def predict_price(values):
    frame = pd.DataFrame([np.asarray(values, dtype=float)], columns=features)
    return float(model.predict(frame)[0])

practice_result = minimize(lambda values: -predict_price(values), x0=baseline.values,
                           bounds=bounds, method="L-BFGS-B")
print("Guided-practice L-BFGS-B result (not a reliable optimum for a forest):")
display(pd.Series(practice_result.x, index=features, name="Value"))
print(f"Predicted price: ${-practice_result.fun:.2f}")

def objective_function(values):
    return -predict_price(np.rint(values))

result = differential_evolution(objective_function, bounds=bounds, seed=42,
                                maxiter=150, popsize=20, polish=False, tol=0.0001)
observed_best = X.iloc[int(np.argmax(model.predict(X)))].to_numpy()
candidates = [np.rint(result.x), observed_best, np.rint(baseline.values)]
optimal_feature_values = max(candidates, key=predict_price)
optimal_price = predict_price(optimal_feature_values)
display(pd.DataFrame({"Feature": features, "Best-found value": optimal_feature_values.astype(int)}))
print(f"Best-found (approximate optimal) predicted price: ${optimal_price:.2f}")


## 7–9. Separate what-if analyses
Interpret “adjust by” as an increase, following the guided practice. Each change starts independently from the cleaned feature means, not from the previous change or optimization result. Fractional means represent the baseline profile used by the guided practice.

In [ ]:
baseline_price = predict_price(baseline.values)
what_if_rows = []
for feature, increase in [("minimum_nights", 14), ("availability_365", 50), ("number_of_reviews", 30)]:
    header(f"What-If Analysis: {feature} + {increase}")
    adjusted = baseline.copy()
    adjusted[feature] += increase
    price = predict_price(adjusted.values)
    display(adjusted.rename("Adjusted value"))
    print(f"Baseline predicted price: ${baseline_price:.2f}")
    print(f"Adjusted predicted price: ${price:.2f}")
    print(f"Change in predicted price: ${price - baseline_price:.2f}")
    what_if_rows.append({"Change": f"{feature} + {increase}", "Predicted price": price,
                         "Difference from baseline": price - baseline_price})
what_if_results = pd.DataFrame(what_if_rows)
display(what_if_results.round(2))


## 10. Evaluate the six scenarios
The specified scenario numbers are absolute values, not increases. Hold each unspecified predictor at its cleaned dataset mean so the scenarios share a consistent baseline.

In [ ]:
header("Evaluate Six Scenarios")
scenarios = {
    "a. 7 nights; 100 reviews": {"minimum_nights": 7, "number_of_reviews": 100},
    "b. 1 night; 10 reviews": {"minimum_nights": 1, "number_of_reviews": 10},
    "c. 2 nights; 100 available days": {"minimum_nights": 2, "availability_365": 100},
    "d. 14 nights; 300 available days": {"minimum_nights": 14, "availability_365": 300},
    "e. 300 available days; 5 reviews": {"availability_365": 300, "number_of_reviews": 5},
    "f. 100 available days; 200 reviews": {"availability_365": 100, "number_of_reviews": 200}
}
scenario_rows = []
for label, settings in scenarios.items():
    values = baseline.copy()
    for feature, value in settings.items():
        values[feature] = value
    price = predict_price(values.values)
    scenario_rows.append({"Scenario": label, **values.to_dict(), "Predicted price": price,
                          "Difference from baseline": price - baseline_price})
scenario_results = pd.DataFrame(scenario_rows).set_index("Scenario")
display(scenario_results.round(2))


## 11. Findings for your lab report
The following cell generates answers from your actual run. Review these and transfer them into the assigned lab report template with your screenshots.

In [ ]:
header("Lab Report Findings")
display(Markdown(f"**a. Null values and correction:** Yes. The original dataset contained {missing_prices} missing price values. I removed those rows because price is the target variable and filling it would invent training labels. I also found {invalid_availability_count} invalid availability value(s) outside 0–365, treated them as missing, and replaced them with the valid median of {feature_medians['availability_365']:g} days. The cleaned modeling columns contain no null values, leaving {len(clean_df):,} records."))
highest = scenario_results["Predicted price"].idxmax()
lowest = scenario_results["Predicted price"].idxmin()
lines = [f"{label}: ${row['Predicted price']:.2f}" for label, row in scenario_results.iterrows()]
display(Markdown("**b. Six-scenario evaluation:** " + "; ".join(lines) + ". "
    + f"The highest prediction is scenario {highest}, at ${scenario_results.loc[highest, 'Predicted price']:.2f}; "
    + f"the lowest is scenario {lowest}, at ${scenario_results.loc[lowest, 'Predicted price']:.2f}. "
    + "These comparisons show how this fitted model responds to the specified combinations while holding unspecified features at their mean values. They are conditional price estimates, not guaranteed prices or causal effects. A higher predicted listing price does not establish higher occupancy, revenue, or profit."))


## Submission checklist
- Replace the student ID in the first Markdown cell and setup cell; update the date if needed.
- Confirm the workbook path in VCASTLE and run all cells from top to bottom.
- Capture the data exploration/cleaning, importances, all three sensitivity plots, optimization definition and results, three what-if outputs, and six-scenario results. Include student ID, date, and time in screenshots.
- Place screenshots and the two written answers in your instructor’s lab report template.
- Save the notebook and submit the report as instructed.

Sources: supplied assignment screenshot and Guided Practice: Performing Prescriptive Analysis Using Python; data: AirbnbListings_v4.xlsx. The optimization improvement and availability cleaning are explicitly documented adaptations.